# Track Labeling Tool

Use this notebook to **manually label launch and landing points** in a column fixture
from `test/fixtures/`. Labels are saved to `test/labels/<stem>.json` and checked by
`test/labels.test.js`: each labeled launch and landing must be within 60 seconds of a
detected segment.

Fixtures are generated by the NorCal-Hike-and-Fly repo (`npm run fixtures` in
`tracklog_handler`), so this notebook needs no IGC/GPX parser.

### Quick-start
1. Install dependencies (first time only): `pip install ipywidgets ipyleaflet plotly anywidget`
2. Run all cells (`Kernel → Restart & Run All`)
3. Set `FIXTURE` in the **Configuration** cell
4. Re-run the **Load** and **UI** cells
5. The sliders start at the current labels (or, if none, at the detected segments).
   Drag them to the correct fixes, then click **Save Labels**
6. Run `npm test` to check the labels against the detector


In [ ]:
import gzip, json, warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings('ignore')

try:
    import ipywidgets as widgets
    import ipyleaflet as ipl
    import plotly.graph_objects as go
    from IPython.display import display
    print("All packages loaded.")
except ImportError as e:
    print(f"Missing: {e}")
    print("Run: pip install ipywidgets ipyleaflet plotly anywidget")
    raise


In [ ]:
# ---- Fixture helpers ----

FIXTURES_DIR = Path("test/fixtures")
LABELS_DIR   = Path("test/labels")


def list_fixtures():
    return sorted(p.name for p in FIXTURES_DIR.glob("*.json.gz"))


def fixture_stem(name):
    return name[:-len(".json.gz")] if name.endswith(".json.gz") else Path(name).stem


def load_fixture(path):
    """Read a gzipped column fixture and return a list of fix dicts plus the golden segments."""
    path = Path(path)
    with gzip.open(path, "rt", encoding="utf8") as fh:
        data = json.load(fh)
    n = len(data["timeMs"])
    for col in ("lat", "lon", "alt"):
        if len(data[col]) != n:
            raise ValueError(f"Column {col} has {len(data[col])} entries, timeMs has {n}")
    fixes = [{
        "timestamp": data["timeMs"][i],
        "time":      datetime.fromtimestamp(data["timeMs"][i] / 1000, tz=timezone.utc).isoformat(),
        "latitude":  data["lat"][i],
        "longitude": data["lon"][i],
        "altitude":  data["alt"][i],
    } for i in range(n)]
    return fixes, data.get("segments", []), data.get("source")


def load_labels(stem):
    """Existing labels for this fixture, or None."""
    path = LABELS_DIR / f"{stem}.json"
    if not path.exists():
        return None
    return json.loads(path.read_text())


print("Fixtures available:")
for name in list_fixtures():
    labeled = "labeled" if (LABELS_DIR / f"{fixture_stem(name)}.json").exists() else "unlabeled"
    print(f"  {name:45s} {labeled}")


## Configuration

Change `FIXTURE` to one of the fixture files listed above.

In [ ]:
# ---- Change this path ----
FIXTURE = "test/fixtures/jerm20260301.json.gz"

LABELS_DIR.mkdir(parents=True, exist_ok=True)
STEM = fixture_stem(Path(FIXTURE).name)

print(f"Fixture    : {FIXTURE}")
print(f"Labels file: {(LABELS_DIR / f'{STEM}.json').resolve()}")


In [ ]:
fixes, golden, source = load_fixture(FIXTURE)
existing = load_labels(STEM)

if len(fixes) < 5:
    raise ValueError(f"Only {len(fixes)} fixes — is the file correct?")

dur_min = (fixes[-1]['timestamp'] - fixes[0]['timestamp']) / 60_000
print(f"Source   : {source}")
print(f"Loaded   : {len(fixes)} fixes")
print(f"Start    : {fixes[0]['time']}")
print(f"End      : {fixes[-1]['time']}")
print(f"Duration : {dur_min:.1f} min")
print(f"Detected : {golden}")
if existing:
    print(f"Labels   : {[(e['launch_idx'], e['landing_idx']) for e in existing['expected']]}")
else:
    print("Labels   : none yet — sliders start at the detected segments")


In [ ]:
# ===================================================================
# Interactive labeling UI
# Run this cell to display the tool. Add one segment per flight.
# ===================================================================

n = len(fixes)
segments = []

# ---- Map ----
lats = [f['latitude']  for f in fixes]
lons = [f['longitude'] for f in fixes]
STRIDE = max(1, n // 500)
m = ipl.Map(center=(sum(lats)/n, sum(lons)/n), zoom=13, scroll_wheel_zoom=True)
m.layout.height = '360px'
m.add(ipl.Polyline(locations=list(zip(lats[::STRIDE], lons[::STRIDE])),
                   color='royalblue', weight=2, opacity=0.8))

# ---- Altitude chart ----
times_s = [(f['timestamp'] - fixes[0]['timestamp']) / 1000 for f in fixes]
alts    = [f.get('altitude') or 0 for f in fixes]

fig = go.FigureWidget()
fig.add_trace(go.Scatter(x=times_s, y=alts, name='Altitude',
                         line=dict(color='steelblue', width=1.5)))
fig.update_layout(
    height=300,
    margin=dict(l=55, r=15, t=30, b=40),
    xaxis_title='Seconds from track start',
    yaxis_title='Altitude (m)',
    showlegend=False,
)

# Detected (golden) segments are drawn as faint dotted lines for reference.
golden_shapes = []
for seg in golden:
    for key, color in (('launch', 'green'), ('landing', 'red')):
        x = times_s[seg[key]]
        golden_shapes.append(dict(type='line', xref='x', yref='paper', x0=x, x1=x, y0=0, y1=1,
                                  line=dict(color=color, width=1, dash='dot'), opacity=0.4))

def redraw_shapes():
    shapes = list(golden_shapes)
    for seg in segments:
        shapes.append(dict(type='line', xref='x', yref='paper',
                           x0=times_s[seg['launch']], x1=times_s[seg['launch']],
                           y0=0, y1=1, line=dict(color='green', width=2, dash='dash')))
        shapes.append(dict(type='line', xref='x', yref='paper',
                           x0=times_s[seg['landing']], x1=times_s[seg['landing']],
                           y0=0, y1=1, line=dict(color='red', width=2, dash='dash')))
    fig.layout.shapes = shapes

# ---- Segment list ----
segments_box = widgets.VBox([])

def rebuild_segments_ui():
    for i, seg in enumerate(segments):
        seg['index_lbl'].value = f"<b>Flight {i + 1}</b>"
    segments_box.children = tuple(seg['container'] for seg in segments)

def make_segment(launch=0, landing=None):
    if landing is None:
        landing = n - 1
    seg = {'launch': launch, 'landing': landing}

    index_lbl   = widgets.HTML(value='')
    launch_lbl  = widgets.HTML(
        value=f"<b style='color:green'>Launch  idx={launch}</b>  {fixes[launch]['time']}")
    landing_lbl = widgets.HTML(
        value=f"<b style='color:red'>Landing idx={landing}</b>  {fixes[landing]['time']}")

    launch_slider = widgets.IntSlider(
        value=launch, min=0, max=n-1, step=1, description='Launch:',
        layout=widgets.Layout(width='95%'),
        style={'description_width': '70px'}, continuous_update=True,
    )
    landing_slider = widgets.IntSlider(
        value=landing, min=0, max=n-1, step=1, description='Landing:',
        layout=widgets.Layout(width='95%'),
        style={'description_width': '70px'}, continuous_update=True,
    )

    launch_marker = ipl.CircleMarker(
        location=(fixes[launch]['latitude'], fixes[launch]['longitude']),
        radius=8, color='green', fill_color='limegreen', fill_opacity=1.0,
    )
    landing_marker = ipl.CircleMarker(
        location=(fixes[landing]['latitude'], fixes[landing]['longitude']),
        radius=8, color='darkred', fill_color='red', fill_opacity=1.0,
    )
    m.add(launch_marker)
    m.add(landing_marker)

    remove_btn = widgets.Button(description='Remove', button_style='danger', icon='trash',
                                layout=widgets.Layout(width='100px'))

    seg.update({
        'index_lbl': index_lbl,
        'launch_lbl': launch_lbl,    'landing_lbl': landing_lbl,
        'launch_slider': launch_slider, 'landing_slider': landing_slider,
        'launch_marker': launch_marker, 'landing_marker': landing_marker,
        'container': widgets.VBox([
            widgets.HBox([index_lbl, widgets.HTML("&nbsp;&nbsp;"), remove_btn]),
            launch_lbl, launch_slider,
            widgets.HTML("<div style='height:4px'></div>"),
            landing_lbl, landing_slider,
            widgets.HTML("<hr style='margin:6px 0; border-color:#ddd'>"),
        ]),
    })

    def on_launch(change):
        idx = change['new']
        seg['launch'] = idx
        f = fixes[idx]
        launch_marker.location = (f['latitude'], f['longitude'])
        launch_lbl.value = f"<b style='color:green'>Launch  idx={idx}</b>  {f['time']}"
        redraw_shapes()

    def on_landing(change):
        idx = change['new']
        seg['landing'] = idx
        f = fixes[idx]
        landing_marker.location = (f['latitude'], f['longitude'])
        landing_lbl.value = f"<b style='color:red'>Landing idx={idx}</b>  {f['time']}"
        redraw_shapes()

    def on_remove(b):
        m.remove(launch_marker)
        m.remove(landing_marker)
        segments.remove(seg)
        rebuild_segments_ui()
        redraw_shapes()

    launch_slider.observe(on_launch,  names='value')
    landing_slider.observe(on_landing, names='value')
    remove_btn.on_click(on_remove)

    segments.append(seg)
    rebuild_segments_ui()
    redraw_shapes()

# ---- Controls ----
add_btn     = widgets.Button(description='+ Add Flight', button_style='info', icon='plus')
desc_box    = widgets.Text(value=(existing or {}).get('description', ''), description='Notes:',
                           placeholder='Optional description',
                           layout=widgets.Layout(width='99%'),
                           style={'description_width': '70px'})
save_btn    = widgets.Button(description='Save Labels', button_style='success', icon='save')
status_html = widgets.HTML(value='')

add_btn.on_click(lambda b: make_segment())

def on_save(b):
    for i, seg in enumerate(segments):
        if seg['launch'] >= seg['landing']:
            status_html.value = (
                f"<span style='color:red'>Error: Flight {i+1} launch must come before landing.</span>")
            return

    labels = {
        "fixture": Path(FIXTURE).name,
        "source": source,
        "description": desc_box.value.strip(),
        "created_at": datetime.now(timezone.utc).isoformat(),
        "expected": [{
            "launch_time":          fixes[seg['launch']]['time'],
            "landing_time":         fixes[seg['landing']]['time'],
            "launch_timestamp_ms":  fixes[seg['launch']]['timestamp'],
            "landing_timestamp_ms": fixes[seg['landing']]['timestamp'],
            "launch_idx":           seg['launch'],
            "landing_idx":          seg['landing'],
        } for seg in sorted(segments, key=lambda s: s['launch'])],
    }

    out_path = LABELS_DIR / f"{STEM}.json"
    out_path.write_text(json.dumps(labels, indent=2) + "\n")
    status_html.value = f"<span style='color:green'>Saved → {out_path}</span>"
    print(f"Saved: {out_path}")

save_btn.on_click(on_save)

# ---- Initial segments: existing labels, else detected segments, else one full-track segment ----
if existing and existing.get('expected'):
    for e in existing['expected']:
        make_segment(e['launch_idx'], e['landing_idx'])
elif golden:
    for seg in golden:
        make_segment(seg['launch'], seg['landing'])
else:
    make_segment()

display(widgets.VBox([
    widgets.HTML("<h3 style='margin:4px 0'>Track Labeling Tool</h3>"),
    widgets.HTML("<span style='color:#666'>Dotted lines on the chart are the detector's current launch (green) and landing (red).</span>"),
    desc_box,
    widgets.HTML("<hr style='margin:6px 0'>"),
    segments_box,
    add_btn,
    widgets.HTML("<hr style='margin:6px 0'>"),
    m, fig,
    widgets.HBox([save_btn, widgets.HTML("&nbsp;&nbsp;"), status_html]),
], layout=widgets.Layout(width='100%')))
